# 03 - Document indexing, semantic search, and evaluation
**Goal:** prepare documents, reuse or generate embeddings, search local Chroma, and inspect results.
Run ingestion once. EDA is not a runtime dependency.

**Storage:** the existing `%LOCALAPPDATA%/Drug_Assist/chroma` database and `drug_documents` collection.
Docker is optional future work. Existing embeddings and review flags are retained.
API-backed cells are disabled by default so **Run All** does not incur API charges.

## 1. Setup and configuration

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

# Works from the project root or its notebooks folder.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").exists() and (path / "data/raw/drug.csv").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the Drug_Assist project folder.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

RAW_PATH = ROOT / "data/raw/drug.csv"
CLEAN_PATH = ROOT / "data/processed/drug_clean.csv"
print("Python:", sys.executable)
print("Project:", ROOT)

Python: c:\Users\suvra_nw8ieuf\AppData\Local\uv-envs\Drug_Assist\Scripts\python.exe
Project: c:\Users\suvra_nw8ieuf\OneDrive\Desktop\Drug_Assist


In [2]:
from drug_assist.data import build_documents
from drug_assist.retrieval import (
    COLLECTION_NAME, EMBEDDING_MODEL, EMBEDDING_DIMENSIONS,
    database_path, openai_client, pending_documents, embed_texts,
    index_missing, flag_for_review, search,
)
import chromadb

# Turn on only the operations you want to run, then rerun their cells.
RUN_INDEXING = False       # Generate and persist missing document embeddings.
RUN_SEARCH = False         # Embed one query and reuse it for both searches.
RUN_EVALUATION = False     # Embed the evaluation queries.

QUERY = "medicines for depression"
CONDITION = "depression"
TOP_K = 3
BATCH_SIZE = 100

## 2. Define and prepare searchable documents (original steps 1–3)
One document is a unique **Condition + Drug + Information** combination.
The 1,753 cleaned rows originally produced 624 searchable documents.
Ratings remain in the CSV because deduplicating this text can combine rows with different ratings.

Schema: `id` (text), `document_text` (text), `metadata` (drug, condition, information,
and review flag). Embeddings are stored separately in Chroma.
The original `drug_N` IDs are retained for compatibility. Changing source order can change IDs;
indexing checks existing text before writing and stops on conflicts.

In [3]:
if not CLEAN_PATH.exists():
    raise FileNotFoundError("Run 01_data_ingestion.ipynb first.")
df_clean = pd.read_csv(CLEAN_PATH, dtype={"Reviews": "Int64"})
from drug_assist.data import validate_clean_data
validate_clean_data(df_clean)
print(f"Loaded {len(df_clean):,} cleaned records")

Loaded 1,753 cleaned records


In [4]:
documents = build_documents(df_clean)
print("Documents prepared:", len(documents))
display(documents[0])

Documents prepared: 624


{'id': 'drug_0',
 'document_text': 'Condition: Acute Bacterial Sinusitis\nDrug: Levofloxacin\nInformation: Levofloxacin is used to treat a variety of bacterial infections. This is a generic drug. The average cash price for 10 Tablet(s), 500mg each of the generic (levofloxacin) is $172.99. You can buy levofloxacin at the discounted price of $47.08 by using the WebMDRx coupon, a savings of 73%. Even if this drug is covered by Medicare or your insurance, we recommend you compare prices. The WebMDRx coupon or cash price may be less than your co-pay.',
 'metadata': {'condition': 'Acute Bacterial Sinusitis',
  'drug': 'Levofloxacin',
  'information': 'Levofloxacin is used to treat a variety of bacterial infections. This is a generic drug. The average cash price for 10 Tablet(s), 500mg each of the generic (levofloxacin) is $172.99. You can buy levofloxacin at the discounted price of $47.08 by using the WebMDRx coupon, a savings of 73%. Even if this drug is covered by Medicare or your insuranc

## 3. Open local Chroma (original step 6)
The model remains `text-embedding-3-small`, with 1,536 dimensions.
The collection uses cosine distance. A smaller distance means closer text similarity.
Reopening this collection does not regenerate embeddings.

In [5]:
db_path = database_path()
chroma_client = chromadb.PersistentClient(path=str(db_path))
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=None,
    configuration={"hnsw": {"space": "cosine"}},
)
pending = pending_documents(collection, documents)
print("Database:", db_path)
print("Stored records:", collection.count())
print("Documents missing from Chroma:", len(pending))

Database: C:\Users\suvra_nw8ieuf\AppData\Local\Drug_Assist\chroma
Stored records: 624
Documents missing from Chroma: 0


## 4. Generate and save only missing embeddings (original steps 4–8)
Set `RUN_INDEXING=True` to populate an empty or incomplete collection.
This sends missing document text to OpenAI and incurs API charges.
Each successful batch is saved immediately so a restart reuses completed work.
Existing records are not overwritten, preserving their review flags.
Generation and insertion are paired here to avoid losing all progress on a kernel restart.

In [6]:
pending = pending_documents(collection, documents)
if not pending:
    print("All documents are already stored; no embedding calls needed.")
elif RUN_INDEXING:
    api_client = openai_client(ROOT)
    index_missing(collection, documents, api_client, batch_size=BATCH_SIZE)
else:
    print(f"Indexing disabled. {len(pending)} documents remain to be indexed.")

All documents are already stored; no embedding calls needed.


## 5. Inspect one saved embedding
This reads Chroma; it makes no OpenAI request.

In [7]:
sample = collection.get(ids=[documents[0]["id"]], include=["embeddings"])
if sample["ids"]:
    vector = sample["embeddings"][0]
    assert len(vector) == EMBEDDING_DIMENSIONS
    print("Dimensions:", len(vector))
    print("First five numbers:", vector[:5])
else:
    print("Index the documents before inspecting an embedding.")

Dimensions: 1536
First five numbers: [-0.05773925 -0.04125977  0.0042305   0.03707886 -0.01739502]


## 6. Inspect and flag data-quality issues
Existing flags are respected. A missing flag or `False` means **not flagged**, not verified.
Inspect the exact record before setting `REVIEW_ID` and `REVIEW_REASON`.
Leave `APPLY_REVIEW_FLAG=False` until you intend to save the flag.
Do not identify a bad record by result position: rankings can change.

In [8]:
flagged = collection.get(where={"needs_review": True}, include=["documents", "metadatas"])
print("Flagged records:", len(flagged["ids"]))
display(pd.DataFrame([
    {"id": record_id, **metadata, "document_text": text}
    for record_id, metadata, text in zip(
        flagged["ids"], flagged["metadatas"], flagged["documents"]
    )
]))

REVIEW_ID = ""             # Copy a specific ID from a search result.
REVIEW_REASON = ""         # Describe the observed inconsistency.
APPLY_REVIEW_FLAG = False
if REVIEW_ID:
    display(collection.get(ids=[REVIEW_ID], include=["documents", "metadatas"]))
if APPLY_REVIEW_FLAG:
    flag_for_review(collection, REVIEW_ID, REVIEW_REASON)
    print("Flag saved:", REVIEW_ID)

Flagged records: 0


""


## 7. Semantic search (original step 9)
Set `RUN_SEARCH=True`, then run this cell. One query embedding is created and reused below.
Known flagged records are excluded. Returned IDs make later review reproducible.
Distance measures similarity, not medical suitability or factual accuracy.

In [9]:
RUN_SEARCH = True
query_vector = "medicines for depression"
results = []
if RUN_SEARCH:
    if collection.count() == 0:
        raise ValueError("Index documents before searching.")
    api_client = openai_client(ROOT)
    query_vector = embed_texts(api_client, [QUERY])[0]
    results = search(collection, query_vector, k=TOP_K)
    display(pd.DataFrame(results))
else:
    print("Search disabled. Set RUN_SEARCH=True in the configuration cell.")

,id,information,condition,drug,distance,document_text
0,drug_177,This medication is used to treat mental/mood p...,depression,Amitriptyline,0.484771,Condition: depression\nDrug: Amitriptyline\nIn...
1,drug_181,This medication is used to treat depression. T...,depression,Imipramine Hcl,0.523238,Condition: depression\nDrug: Imipramine Hcl\nI...
2,drug_178,This medication is used to treat mental/mood p...,depression,Nortriptyline,0.523972,Condition: depression\nDrug: Nortriptyline\nIn...


## 8. Metadata filtering (original step 10)
Restrict results to the exact condition label while still excluding review flags.

In [10]:
if query_vector is not None:
    filtered_results = search(collection, query_vector, k=TOP_K, condition=CONDITION)
    display(pd.DataFrame(filtered_results))
else:
    print("Run the semantic-search cell first.")

,id,condition,drug,information,distance,document_text
0,drug_177,depression,Amitriptyline,This medication is used to treat mental/mood p...,0.484771,Condition: depression\nDrug: Amitriptyline\nIn...
1,drug_181,depression,Imipramine Hcl,This medication is used to treat depression. T...,0.523238,Condition: depression\nDrug: Imipramine Hcl\nI...
2,drug_178,depression,Nortriptyline,This medication is used to treat mental/mood p...,0.523972,Condition: depression\nDrug: Nortriptyline\nIn...


## 9. Retrieval evaluation (original step 11)
Use several queries with expected condition labels. This is a **condition-match proxy**,
not a full assessment of relevance or factual accuracy. Queries are not filtered to their
expected condition; doing so would make this check automatic.
`condition_precision_at_k` divides by the requested K; fewer returned records cannot inflate it.
Review descriptions manually as well, and expand the cases beyond these starter examples.

In [11]:
RUN_EVALUATION = True
evaluation_cases = [
    {"query": "medicines for hypertension", "expected_condition": "hypertension"},
    {"query": "medicines for depression", "expected_condition": "depression"},
]
if RUN_EVALUATION:
    if collection.count() == 0:
        raise ValueError("Index documents before evaluation.")
    known_conditions = set(df_clean["Condition"].dropna())
    for case in evaluation_cases:
        if case["expected_condition"] not in known_conditions:
            raise ValueError(f"Unknown condition label: {case['expected_condition']}")
    api_client = openai_client(ROOT)
    vectors = embed_texts(api_client, [case["query"] for case in evaluation_cases])
    evaluation = []
    for case, vector in zip(evaluation_cases, vectors):
        matches = search(collection, vector, k=TOP_K)
        correct = sum(item["condition"] == case["expected_condition"] for item in matches)
        evaluation.append({
            **case, "returned": len(matches), "correct_condition": correct,
            "condition_precision_at_k": correct / TOP_K,
            "retrieved_ids": [item["id"] for item in matches],
        })
    display(pd.DataFrame(evaluation))
else:
    print("Evaluation disabled. Set RUN_EVALUATION=True to run the test queries.")

,query,expected_condition,returned,correct_condition,condition_precision_at_k,retrieved_ids
0,medicines for hypertension,hypertension,3,3,1.0,"[drug_543, drug_481, drug_477]"
1,medicines for depression,depression,3,3,1.0,"[drug_177, drug_181, drug_178]"


In [12]:
evaluation_df = pd.DataFrame(evaluation)
evaluation_df["Precision (%)"] = (
    evaluation_df["condition_precision_at_k"]
    .map(lambda score: f"{score:.1%}")
)
display(evaluation_df)

,query,expected_condition,returned,correct_condition,condition_precision_at_k,retrieved_ids,Precision (%)
0,medicines for hypertension,hypertension,3,3,1.0,"[drug_543, drug_481, drug_477]",100.0%
1,medicines for depression,depression,3,3,1.0,"[drug_177, drug_181, drug_178]",100.0%


## Next: improve retrieval before RAG
Inspect mismatched descriptions, save flags, and broaden the evaluation queries.
LangChain/LlamaIndex and answer generation belong to later work; they are not required
for this direct Chroma workflow. Reuse this database when adding RAG.